# Exploratory Data Analysis

**English training data:** Davidson et al. 2017 (`tdavidson/hate_speech_offensive`)  
**Urdu evaluation data:** Roman Urdu Hate Speech (`community-datasets/roman_urdu_hate_speech`)

This notebook:
1. Loads both datasets and reports class distributions
2. Analyses text length distributions
3. Shows top word frequencies per class
4. Plots label correlations
5. Saves figures to `outputs/figures/` for the mid-report

In [ ]:
import sys, os
os.chdir('..')   # run from project root
sys.path.insert(0, '.')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from collections import Counter
import re

from src.data_loader import load_hateval_tsv

sns.set_theme(style='whitegrid', palette='Set2')
os.makedirs('outputs/figures', exist_ok=True)
print('Ready.')

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────────
en_train = load_hateval_tsv('data/raw/hateval2019_en_train.tsv')
en_dev   = load_hateval_tsv('data/raw/hateval2019_en_dev.tsv')

ur_train = load_hateval_tsv('data/raw/hateval2019_ur_train.tsv') if os.path.exists('data/raw/hateval2019_ur_train.tsv') else None
ur_dev   = load_hateval_tsv('data/raw/hateval2019_ur_dev.tsv')   if os.path.exists('data/raw/hateval2019_ur_dev.tsv')   else None
ur_test  = load_hateval_tsv('data/raw/hateval2019_ur_test.tsv')  if os.path.exists('data/raw/hateval2019_ur_test.tsv')  else None

print(f'EN train: {len(en_train):,}   EN dev: {len(en_dev):,'}
for tag, df in [('UR train', ur_train), ('UR dev', ur_dev), ('UR test', ur_test)]:
    if df is not None: print(f'{tag}: {len(df):,}')

In [ ]:
# ── Class distribution table ───────────────────────────────────────────────
splits = {'EN train': en_train, 'EN dev': en_dev}
if ur_train is not None: splits['UR train'] = ur_train
if ur_dev   is not None: splits['UR dev']   = ur_dev
if ur_test  is not None: splits['UR test']  = ur_test

rows = []
for name, df in splits.items():
    if df is None or 'hs' not in df.columns: continue
    total = len(df)
    n_hate = int(df['hs'].sum())
    rows.append({'Split': name, 'Total': total,
                 'Hate': n_hate, 'Hate (%)': 100*n_hate/total,
                 'Non-hate': total-n_hate, 'Non-hate (%)': 100*(total-n_hate)/total})

stats_df = pd.DataFrame(rows)
display(stats_df.round(1))

In [ ]:
# ── Figure 1: Class distribution ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(stats_df))
w = 0.35
ax.bar([i-w/2 for i in x], stats_df['Hate (%)'],     w, label='Hate',     color='#e74c3c', alpha=0.85)
ax.bar([i+w/2 for i in x], stats_df['Non-hate (%)'], w, label='Non-hate', color='#2ecc71', alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels(stats_df['Split'])
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylabel('Percentage of samples')
ax.set_title('Class Distribution — English (Davidson 2017) & Urdu (Roman Urdu HS)')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/figures/class_distribution.png', dpi=150)
plt.show()
print('Saved → outputs/figures/class_distribution.png')

In [ ]:
# ── Figure 2: Token length distribution ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, df, title, color in [
    (axes[0], en_train, 'English Training Set (Davidson 2017)', '#3498db'),
    (axes[1], ur_train if ur_train is not None else en_train,
     'Urdu Training Set (Roman Urdu HS)', '#9b59b6'),
]:
    lens = df['text'].str.split().str.len()
    ax.hist(lens, bins=30, color=color, alpha=0.8, edgecolor='white')
    ax.set_xlabel('Tokens (whitespace split)')
    ax.set_ylabel('Count')
    ax.set_title(f'{title}\nmedian={lens.median():.0f}, p95={lens.quantile(0.95):.0f}')

plt.suptitle('Tweet/Post Length Distribution', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/length_distribution.png', dpi=150)
plt.show()

In [ ]:
# ── Top 20 words in hate vs. non-hate (English) ───────────────────────────
STOPWORDS = {'the','a','an','is','are','was','were','to','of','and','in','it',
             'i','you','we','they','he','she','that','this','for','on','with',
             'be','have','do','at','by','not','rt','amp','http','https','co'}

def top_words(df, label, n=20):
    texts = df[df['hs'] == label]['text'].str.lower()
    tokens = [w for t in texts for w in re.findall(r'\b[a-z]{3,}\b', t) if w not in STOPWORDS]
    return Counter(tokens).most_common(n)

print('Top 20 words in HATE tweets (English):')
for w, c in top_words(en_train, 1, 20):
    print(f'  {w:<20} {c}')

print('\nTop 20 words in NON-HATE tweets (English):')
for w, c in top_words(en_train, 0, 20):
    print(f'  {w:<20} {c}')

In [ ]:
# ── Urdu vocabulary peek ───────────────────────────────────────────────────
if ur_train is not None:
    print('Sample HATE posts (Roman Urdu):')
    for t in ur_train[ur_train['hs']==1]['text'].sample(5, random_state=42):
        print(f'  {t[:120]}')
    print('\nSample NON-HATE posts (Roman Urdu):')
    for t in ur_train[ur_train['hs']==0]['text'].sample(5, random_state=42):
        print(f'  {t[:120]}')

In [ ]:
print('EDA complete. Figures saved to outputs/figures/')